# Exploratory Data Analysis

End-to-end EDA notebook for the AI Enterprise Workflow capstone.
Drives the full ingestion pipeline via `ingest(force=True)`, then
analyses the aggregated daily revenue time series using STL
decomposition, an Augmented Dickey-Fuller stationarity test, and
autocorrelation / partial-autocorrelation plots.

**Prerequisites:** raw invoice JSON files must exist under `data/input/`.

In [ ]:
import os
import pathlib

# Ensure the kernel runs from the repo root so cfg paths resolve correctly.
# VS Code defaults the kernel CWD to the notebook folder (notebooks/).
_cwd = pathlib.Path.cwd()
if _cwd.name == "notebooks":
    os.chdir(_cwd.parent)

In [ ]:
import sys
import pathlib

sys.path.insert(0, str(pathlib.Path.cwd() / "src"))

import pandas as pd
from matplotlib import pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

from ai_enterprise_workflow.core.config import cfg
from ai_enterprise_workflow.ingestion import ingest

%matplotlib inline

## 1. Ingestion

Run the full ingestion pipeline. `force=True` always re-reads the raw
JSON files and overwrites any existing output CSVs.

In [ ]:
ingest(force=True)

In [ ]:
revenue_total = pd.read_csv(cfg.directory_output / "4 revenue_total.csv")
revenue_total.head()

## 2. Revenue over time

In [ ]:
revenue_total.plot(x="date", y="revenue", legend=False)
plt.show()

## 3. STL decomposition — trend and seasonal components (LOESS)

In [ ]:
stl = STL(revenue_total["revenue"], period=30)
stl.fit().plot()
plt.show()

## 4. Stationarity — Augmented Dickey-Fuller test

Tests whether the revenue time series has a unit root (non-stationary).
A *p*-value < 0.05 rejects the null hypothesis of a unit root.

In [ ]:
adfuller_results = adfuller(revenue_total["revenue"])
if adfuller_results[1] < 0.05:
    print("Data is stationary (p =", round(adfuller_results[1], 6), ")")
else:
    print("Data is not stationary (p =", round(adfuller_results[1], 6), ")")

## 5. Autocorrelation and partial autocorrelation

In [ ]:
plot_acf(revenue_total["revenue"])
plt.show()

plot_pacf(revenue_total["revenue"])
plt.show()